In [2]:
import os
import sys
word_dir = '/Users/lxx/Documents/codes/st-missing-fill'
os.chdir(word_dir)
sys.path.append(word_dir)


In [4]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


from src.data.load import load_data
from src.data.misser import simulate_missingness
from src.models.baselines import locf_imputer
from src.evaluate import evaluate_rmse

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
%config InlineBackend.figure_format = 'retina'

Using device: mps


# 以下是测试部分

数据集切分：

训练集：验证集：测试集 = 4：2：1

In [5]:
window_size = step_size = 6 * 24


In [ ]:
# 1 读取数据
ground_X, ground_y, all_stations, all_stations_cluster, vars = load_data()
S, T = ground_y.shape

print(f'ground_X shape: {ground_X.shape}')
print(f'ground_y shape: {ground_y.shape}')


# 2 切分数据集
train_size = len(pd.date_range(start='2023-01-01 00:00:00', end='2023-12-31 23:50:00', freq='10min'))
val_size = len(pd.date_range(start='2024-01-01 00:00:00', end='2024-06-30 23:50:00', freq='10min'))

ground_X_train, ground_X_val, ground_X_test = ground_X[..., :train_size], ground_X[..., train_size:train_size+val_size], ground_X[..., train_size+val_size:]
ground_y_train, ground_y_val, ground_y_test = ground_y[:, :train_size], ground_y[:, train_size:train_size+val_size], ground_y[:, train_size+val_size:]

# 3 转化成需要的输入格式
ground_y_train = ground_y_train.reshape(-1, window_size)
ground_y_train = ground_y_train[..., np.newaxis]

ground_y_val = ground_y_val.reshape(-1, window_size)
ground_y_val = ground_y_val[..., np.newaxis]

ground_y_test = ground_y_test.reshape(-1, window_size)
ground_y_test = ground_y_test[..., np.newaxis]

print(f'ground_y_train shape after reshape: {ground_y_train.shape}')
print(f'ground_y_val shape after reshape: {ground_y_val.shape}')
print(f'ground_y_test shape after reshape: {ground_y_test.shape}')


Missing rate in ground_y: 0.27%
Missing rate in ground_X: 0.18%
ground_X shape: (112, 7, 105264)
ground_y shape: (112, 105264)


ground_y_train shape after reshape: (81872, 144, 1)
ground_y_val shape after reshape: (20384, 144, 1)
ground_y_test shape after reshape: (20608, 144, 1)


In [ ]:
from pypots.imputation import SAITS

model = SAITS(n_steps=ground_y_train.shape[1], n_features=ground_y_train.shape[2], n_layers=4, d_model=32, n_heads=4, d_k=8,
                d_v=32, d_ffn=64, epochs=100, verbose=False)

train_set = {'X': ground_y_train}
val_set = {'X': ground_y_val, 'X_ori': ground_y_val}

model.fit(train_set)
imputed_val = model.predict(ground_y_val)

In [6]:
print('真实值: ground_y', ground_y.shape)
print('掩码值: mask', mask.shape) # 0 表示缺失值
print('掩码后的值: y_masked', y_masked.shape)

真实值: ground_y (112, 105264)
掩码值: mask (112, 105264)
掩码后的值: y_masked (112, 105264, 1)
